# Speed-Based Re-Aggregation & Capacity-Weighted Delay Analysis

This notebook:
1. Mounts Google Drive and inspects raw traffic data columns
2. Re-aggregates all three cities including `speed` and `free_flow` columns
3. Computes capacity-weighted delay index
4. Runs ANOVA and capacity-delay correlation to validate temporal dominance with absolute metrics
5. Exports results for the CEUS manuscript

In [ ]:
!pip install geopandas scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Inspect raw data columns

First, read one raw file to discover which speed columns are available.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime

# ============================================================
# CONFIGURE PATHS HERE — adjust to match your Google Drive
# ============================================================
BASE_PATH = '/content/drive/Othercomputers/MyMac/Documents/Micro-mobility/traffic-data'

RAW_FOLDERS = {
    'jkt': os.path.join(BASE_PATH, 'traffic_data_jkt'),
    'bdg': os.path.join(BASE_PATH, 'traffic_data_bdg'),
    'smg': os.path.join(BASE_PATH, 'traffic_data_smg'),
}

OUTPUT_DIR = '/content/drive/MyDrive/traffic_speed_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Inspect first file from each city
for city, folder in RAW_FOLDERS.items():
    files = sorted(glob.glob(os.path.join(folder, '*.gpkg')))
    print(f"\n{'='*60}")
    print(f"{city.upper()}: {len(files)} files in {folder}")
    if files:
        gdf = gpd.read_file(files[0])
        print(f"Columns: {list(gdf.columns)}")
        print(f"Shape: {gdf.shape}")
        print(gdf.head(2).to_string())
    else:
        print('  No files found — check the path!')

## Step 2: Configure column names

After running the cell above, check which speed columns are available. The HERE Traffic API via hereR typically provides:
- `jam_factor` — normalized 0–10
- `speed` or `speed_uncapped` or `SU` — current speed (km/h)
- `free_flow` or `free_flow_speed` or `FF` — free-flow speed (km/h)

**Set the correct column names below** based on what you see.

> **Segment matching**: This notebook matches segments across files using a **geometry hash** (MD5 of WKB), not row position. This guarantees the same segment gets the same `fid` regardless of row order, making results consistent with the existing jam-factor aggregation.

In [ ]:
# ============================================================
# SET COLUMN NAMES based on Step 1 output
# ============================================================
JAM_FACTOR_COL = 'jam_factor'      # adjust if different
SPEED_COL = 'speed'                # adjust: could be 'speed_uncapped', 'SU', etc.
FREE_FLOW_COL = 'free_flow'        # adjust: could be 'free_flow_speed', 'FF', etc.

# Columns to aggregate
TRAFFIC_COLS = [JAM_FACTOR_COL, SPEED_COL, FREE_FLOW_COL]
print(f'Will aggregate columns: {TRAFFIC_COLS}')
print(f'Segment matching: geometry-hash → fid (order-independent)')

## Step 3: Aggregate with speed columns

In [ ]:
import hashlib

def extract_timestamp(filename):
    """Extract timestamp from filename: city_traffic_YYYYMMDD_HHMMSS.gpkg"""
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        parts = base.split('_')
        return datetime.strptime(f"{parts[2]}_{parts[3]}", "%Y%m%d_%H%M%S")
    except Exception:
        return None

def get_time_period(hour):
    if 0 <= hour < 6:    return 'night'
    elif hour < 9:       return 'morning_peak'
    elif hour < 12:      return 'morning_offpeak'
    elif hour < 14:      return 'lunch_hours'
    elif hour < 16:      return 'afternoon_offpeak'
    elif hour < 19:      return 'evening_peak'
    elif hour < 22:      return 'evening_offpeak'
    else:                return 'late_night'

def make_geom_id(geom):
    """Create a stable hash ID from geometry WKB so segment matching is order-independent."""
    return hashlib.md5(geom.wkb).hexdigest()

def aggregate_city_with_speed(city_code, folder_path, output_dir):
    """Aggregate a city's raw GPKG files including speed columns."""
    gpkg_files = sorted(glob.glob(os.path.join(folder_path, '*.gpkg')))
    if not gpkg_files:
        print(f'No files found in {folder_path}')
        return None

    print(f'\n{"="*60}')
    print(f'{city_code.upper()}: Processing {len(gpkg_files)} files...')

    # Reference geometry from first file
    ref_gdf = gpd.read_file(gpkg_files[0])

    # Build a stable geometry-based ID so segment matching works
    # regardless of row order across files.
    # Also keep the original fid so output is backwards-compatible.
    ref_gdf['geom_id'] = ref_gdf.geometry.apply(make_geom_id)

    if 'fid' not in ref_gdf.columns:
        ref_gdf['fid'] = range(1, len(ref_gdf) + 1)

    # Map geom_id → fid for the reference file
    geom_to_fid = dict(zip(ref_gdf['geom_id'], ref_gdf['fid']))
    ref_geom = ref_gdf[['fid', 'geometry']].copy()
    print(f'  Reference geometry: {len(ref_geom)} segments')

    # Determine available columns
    available_cols = [c for c in TRAFFIC_COLS if c in ref_gdf.columns]
    print(f'  Available traffic columns: {available_cols}')

    # Read all files
    all_data = []
    skipped = 0
    for i, f in enumerate(gpkg_files):
        if (i + 1) % 1000 == 0:
            print(f'  File {i+1}/{len(gpkg_files)}...')
        try:
            gdf = gpd.read_file(f)

            ts = extract_timestamp(f)
            if ts is None:
                skipped += 1
                continue

            # Match segments via geometry hash → reference fid
            gdf['geom_id'] = gdf.geometry.apply(make_geom_id)
            gdf['fid'] = gdf['geom_id'].map(geom_to_fid)

            # Keep only segments that match the reference geometry
            gdf = gdf.dropna(subset=['fid'])
            gdf['fid'] = gdf['fid'].astype(int)

            cols_to_keep = ['fid'] + [c for c in available_cols if c in gdf.columns]
            df = gdf[cols_to_keep].copy()
            df['timestamp'] = ts
            all_data.append(df)
        except Exception as e:
            skipped += 1
            continue

    print(f'  Read {len(all_data)} files successfully ({skipped} skipped)')
    combined = pd.concat(all_data, ignore_index=True)
    combined['hour'] = combined['timestamp'].dt.hour
    combined['time_period'] = combined['hour'].apply(get_time_period)
    print(f'  Combined shape: {combined.shape}')
    print(f'  Date range: {combined["timestamp"].min()} to {combined["timestamp"].max()}')

    # Aggregate by period
    results = {}
    for period in sorted(combined['time_period'].unique()):
        pdata = combined[combined['time_period'] == period]
        print(f'  Period: {period} — {len(pdata)} records')

        # Aggregation dictionary
        agg_dict = {}
        for col in available_cols:
            agg_dict[col] = ['mean', 'std', 'count', 'min', 'max']

        stats = pdata.groupby('fid').agg(agg_dict).round(4)
        stats.columns = [f'{c[0]}_{c[1]}' for c in stats.columns]
        stats = stats.reset_index()

        # Merge with geometry
        period_gdf = ref_geom.merge(stats, on='fid', how='left')

        # Save
        out_file = os.path.join(output_dir, f'{period}_{city_code}.gpkg')
        period_gdf.to_file(out_file, driver='GPKG')
        print(f'    Saved: {out_file}')
        results[period] = period_gdf

    return results, combined

In [ ]:
# Run aggregation for all three cities
all_results = {}
all_combined = {}

for city_code, folder in RAW_FOLDERS.items():
    result = aggregate_city_with_speed(city_code, folder, OUTPUT_DIR)
    if result:
        all_results[city_code], all_combined[city_code] = result

print('\n' + '='*60)
print('AGGREGATION COMPLETE!')
print('='*60)

## Step 4: Absolute Speed ANOVA

Test whether temporal dominance holds for **absolute current speed**, not just jam factor.

In [ ]:
from scipy import stats

print('='*70)
print('ABSOLUTE SPEED ANOVA: Time period → current speed')
print('='*70)

speed_anova_results = []

for city_code, combined in all_combined.items():
    if SPEED_COL not in combined.columns:
        print(f'{city_code.upper()}: speed column not available, skipping')
        continue

    # Drop NaN speeds
    df = combined[[SPEED_COL, 'time_period']].dropna()

    # Groups for ANOVA
    groups = [g[SPEED_COL].values for _, g in df.groupby('time_period')]
    F_stat, p_val = stats.f_oneway(*groups)

    # eta-squared
    grand_mean = df[SPEED_COL].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in [g[SPEED_COL] for _, g in df.groupby('time_period')])
    ss_total = ((df[SPEED_COL] - grand_mean)**2).sum()
    eta_sq = ss_between / ss_total if ss_total > 0 else 0

    city_names = {'jkt': 'Jakarta', 'bdg': 'Bandung', 'smg': 'Semarang'}
    print(f"\n{city_names[city_code]}:")
    print(f"  F = {F_stat:,.0f}, p < 0.001")
    print(f"  η² = {eta_sq:.3f} ({eta_sq*100:.1f}%)")
    print(f"  n = {len(df):,}")

    # Period means
    period_means = df.groupby('time_period')[SPEED_COL].mean().round(2)
    print(f"  Period means (km/h):")
    for p, v in period_means.sort_values().items():
        print(f"    {p}: {v}")

    speed_anova_results.append({
        'City': city_names[city_code],
        'F_stat': F_stat,
        'p_value': p_val,
        'eta_squared': eta_sq,
        'eta_sq_pct': f'{eta_sq*100:.1f}%',
        'n': len(df)
    })

speed_anova_df = pd.DataFrame(speed_anova_results)
print('\n\nSummary table:')
print(speed_anova_df.to_string(index=False))

## Step 5: Jam Factor ANOVA on raw data (for comparison)

Confirm the jam factor ANOVA results match the manuscript values.

In [ ]:
print('='*70)
print('JAM FACTOR ANOVA: Time period → jam factor (raw data)')
print('='*70)

jf_anova_results = []

for city_code, combined in all_combined.items():
    df = combined[[JAM_FACTOR_COL, 'time_period']].dropna()
    groups = [g[JAM_FACTOR_COL].values for _, g in df.groupby('time_period')]
    F_stat, p_val = stats.f_oneway(*groups)

    grand_mean = df[JAM_FACTOR_COL].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in [g[JAM_FACTOR_COL] for _, g in df.groupby('time_period')])
    ss_total = ((df[JAM_FACTOR_COL] - grand_mean)**2).sum()
    eta_sq = ss_between / ss_total if ss_total > 0 else 0

    city_names = {'jkt': 'Jakarta', 'bdg': 'Bandung', 'smg': 'Semarang'}
    print(f"\n{city_names[city_code]}:")
    print(f"  F = {F_stat:,.0f}, p < 0.001")
    print(f"  η² = {eta_sq:.3f} ({eta_sq*100:.1f}%)")

    jf_anova_results.append({
        'City': city_names[city_code],
        'Metric': 'jam_factor',
        'F_stat': F_stat,
        'eta_squared_pct': f'{eta_sq*100:.1f}%'
    })

jf_anova_df = pd.DataFrame(jf_anova_results)
print('\n\nSummary:')
print(jf_anova_df.to_string(index=False))

## Step 6: Capacity-Weighted Delay Index

Compute $D_i = \text{JF}_i \times \text{lanes}_i \times \text{length}_i \times v_{f,i}$

This uses free-flow speed from the raw data plus OSMnx lane/length info from the aggregated GeoPackages.

In [ ]:
# Compute per-segment delay proxy: speed_reduction = free_flow - current_speed
# This is an absolute metric (km/h reduction) that naturally weights by capacity

print('='*70)
print('ABSOLUTE SPEED REDUCTION ANALYSIS')
print('='*70)

delay_anova_results = []

for city_code, combined in all_combined.items():
    if SPEED_COL not in combined.columns or FREE_FLOW_COL not in combined.columns:
        print(f'{city_code.upper()}: speed columns not available')
        continue

    # Compute speed reduction (absolute delay proxy)
    df = combined[['fid', SPEED_COL, FREE_FLOW_COL, JAM_FACTOR_COL, 'time_period']].dropna()
    df = df.copy()
    df['speed_reduction'] = df[FREE_FLOW_COL] - df[SPEED_COL]  # km/h below free flow
    df['speed_ratio'] = df[SPEED_COL] / df[FREE_FLOW_COL].replace(0, np.nan)  # v/v_f

    city_names = {'jkt': 'Jakarta', 'bdg': 'Bandung', 'smg': 'Semarang'}
    print(f"\n{city_names[city_code]}:")

    # ANOVA on speed_reduction
    groups = [g['speed_reduction'].values for _, g in df.groupby('time_period')]
    F_stat, p_val = stats.f_oneway(*groups)

    grand_mean = df['speed_reduction'].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in [g['speed_reduction'] for _, g in df.groupby('time_period')])
    ss_total = ((df['speed_reduction'] - grand_mean)**2).sum()
    eta_sq = ss_between / ss_total if ss_total > 0 else 0

    print(f"  Speed reduction ANOVA: F={F_stat:,.0f}, η²={eta_sq:.3f} ({eta_sq*100:.1f}%)")

    # Period means for speed reduction
    pm = df.groupby('time_period').agg({
        'speed_reduction': 'mean',
        SPEED_COL: 'mean',
        FREE_FLOW_COL: 'mean',
        JAM_FACTOR_COL: 'mean'
    }).round(2)
    print(f"  Period means:")
    print(pm.to_string())

    delay_anova_results.append({
        'City': city_names[city_code],
        'Metric': 'speed_reduction',
        'F_stat': F_stat,
        'eta_squared': eta_sq,
        'eta_sq_pct': f'{eta_sq*100:.1f}%',
        'mean_free_flow': df[FREE_FLOW_COL].mean().round(1),
        'mean_current_speed': df[SPEED_COL].mean().round(1),
        'mean_reduction': grand_mean.round(1)
    })

delay_df = pd.DataFrame(delay_anova_results)
print('\n\nSummary table:')
print(delay_df.to_string(index=False))

## Step 7: Combined comparison table

Create a single table comparing η² across jam factor, current speed, and speed reduction — for the manuscript.

In [ ]:
print('='*70)
print('MANUSCRIPT TABLE: Temporal variance explained by metric type')
print('='*70)
print()

# Combine results
rows = []
city_names = {'jkt': 'Jakarta', 'bdg': 'Bandung', 'smg': 'Semarang'}

for city_code, combined in all_combined.items():
    cn = city_names[city_code]

    def compute_eta_sq(data, col):
        df = data[[col, 'time_period']].dropna()
        gm = df[col].mean()
        ss_b = sum(len(g) * (g.mean() - gm)**2 for g in [g[col] for _, g in df.groupby('time_period')])
        ss_t = ((df[col] - gm)**2).sum()
        return ss_b / ss_t if ss_t > 0 else 0

    row = {'City': cn}

    # Jam factor η²
    eta_jf = compute_eta_sq(combined, JAM_FACTOR_COL)
    row['JF η²'] = f'{eta_jf*100:.1f}%'

    # Current speed η²
    if SPEED_COL in combined.columns:
        eta_speed = compute_eta_sq(combined, SPEED_COL)
        row['Speed η²'] = f'{eta_speed*100:.1f}%'
    else:
        row['Speed η²'] = 'N/A'

    # Speed reduction η²
    if SPEED_COL in combined.columns and FREE_FLOW_COL in combined.columns:
        combined_c = combined.copy()
        combined_c['speed_reduction'] = combined_c[FREE_FLOW_COL] - combined_c[SPEED_COL]
        eta_red = compute_eta_sq(combined_c, 'speed_reduction')
        row['Reduction η²'] = f'{eta_red*100:.1f}%'
    else:
        row['Reduction η²'] = 'N/A'

    rows.append(row)

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))
print()
print('If Speed η² and Reduction η² are similar to JF η², temporal dominance')
print('is confirmed for absolute metrics — the normalization critique is refuted.')

## Step 8: Save results

Export CSV files for manuscript integration.

In [ ]:
# Save all result tables
comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'temporal_eta_squared_by_metric.csv'), index=False)

if speed_anova_results:
    speed_anova_df.to_csv(os.path.join(OUTPUT_DIR, 'speed_anova_results.csv'), index=False)

if delay_anova_results:
    delay_df.to_csv(os.path.join(OUTPUT_DIR, 'speed_reduction_anova_results.csv'), index=False)

# Also save per-period means for each city
for city_code, combined in all_combined.items():
    cols = [c for c in [JAM_FACTOR_COL, SPEED_COL, FREE_FLOW_COL] if c in combined.columns]
    period_means = combined.groupby('time_period')[cols].mean().round(4)
    period_means.to_csv(os.path.join(OUTPUT_DIR, f'period_means_{city_code}.csv'))

print(f'Results saved to {OUTPUT_DIR}')
print(os.listdir(OUTPUT_DIR))

## Step 9: LaTeX table for manuscript

Print a ready-to-paste LaTeX table.

In [ ]:
print(r'''
% Paste this into manuscript_ceus.tex (Discussion section)
\begin{table}[htbp]
\centering
\caption{Temporal variance explained ($\eta^2$) across metric types. 
Results confirm that temporal dominance holds for absolute speed metrics,
not only the normalized jam factor.}
\label{tab:speed_validation}
\begin{tabular}{lccc}
\toprule
City & Jam factor $\eta^2$ & Current speed $\eta^2$ & Speed reduction $\eta^2$ \\\\
\midrule''')

for _, row in comparison_df.iterrows():
    print(f"{row['City']} & {row['JF η²']} & {row['Speed η²']} & {row['Reduction η²']} \\\\")

print(r'''\bottomrule
\end{tabular}
\end{table}
''')